# Elo Rating Engine

Tujuan:

Notebook ini bertujuan membangun **Elo Rating Engine** yang digunakan untuk menghitung kekuatan relatif setiap tim nasional berdasarkan riwayat pertandingan internasional.
Berbeda dengan *historical feature engineering* yang menghasilkan statistik sederhana seperti *win rate* atau rata-rata gol, Elo Rating merupakan sistem pemeringkatan dinamis yang memperbarui kekuatan setiap tim setelah setiap pertandingan selesai.

Seluruh proses perhitungan dilakukan secara **kronologis**, sehingga rating yang digunakan pada suatu pertandingan hanya berasal dari informasi yang tersedia sebelum pertandingan tersebut dimainkan. Pendekatan ini memastikan bahwa proses pembangunan fitur tetap bebas dari *future data leakage* dan merepresentasikan kondisi prediksi di dunia nyata (*real-world prediction*).
Output utama dari notebook ini adalah histori Elo Rating setiap tim yang selanjutnya akan digunakan pada tahap **Elo Feature Engineering** untuk membangun fitur-fitur prediktif seperti `home_elo`, `away_elo`, dan `elo_difference`.

## 1. Loading

Pertama, kita akan memuat dataset hasil *data preprocessing* yang akan digunakan sebagai dasar perhitungan Elo Rating.
Pada notebook ini hanya beberapa kolom utama yang diperlukan, yaitu informasi tanggal pertandingan, nama kedua tim, serta skor akhir pertandingan.

In [1]:
import pandas as pd
from collections import defaultdict

In [2]:
results = pd.read_csv("../data/interim/results_clean.csv")

In [ ]:
# AMBIL YANG DIBUTUHKAN
results = results[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
]

In [ ]:
# KONVERSI DATE
results["date"] = pd.to_datetime(results["date"])

In [ ]:
# SORT BY DATE
results = results.sort_values("date").reset_index(drop=True)

In [ ]:
# LIHAT
results.head()

,date,home_team,away_team,home_score,away_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0


Dataset berhasil dimuat dan hanya menyisakan kolom-kolom yang diperlukan untuk membangun Elo Rating. Seluruh pertandingan kemudian diurutkan berdasarkan tanggal guna memastikan proses pembaruan rating mengikuti urutan kronologis. Langkah ini sangat penting karena Elo Rating merupakan sistem yang bergantung pada hasil pertandingan sebelumnya. Dengan demikian, setiap pembaruan rating hanya memanfaatkan informasi yang memang telah tersedia pada saat pertandingan berlangsung.